# edits

> The spine-edit operation vocabulary (`prune` / `replace_text` / `boundary_shift`) + supersession resolution + the effective-view projection. These are generic operations on any NEXT-chained text spine; correction workflows carry them in overlay-node payloads, and the projection interprets them at read time (migrates correction-core C11/C16 onto the layer).

In [ ]:
#| default_exp edits

In [ ]:
#| export
from dataclasses import dataclass, field
from typing import Any, Dict, Iterable, List, Set, Tuple

In [ ]:
#| export
# Reserved spine-edit operation vocabulary (reserve-enum-values-up-front):
# boundary_shift is locked in NOW per the where-graph-begins resolution even
# though no driver produces it yet — the persisted decision preserves the
# alignment-error-vs-transcription-error distinction.
EDIT_OPS = ("prune", "replace_text", "boundary_shift")

In [ ]:
#| export
class SpineEditError(ValueError):
    """A spine edit could not be validated or applied (loud, never silent)."""
    pass

In [ ]:
#| export
@dataclass
class SpineUnit:
    """Minimal projection unit: one spine position with its effective text."""
    id: str    # Layer-0 segment node id
    text: str  # Effective text at this position

In [ ]:
#| export
@dataclass
class SpineEdit:
    """One spine-edit decision, as carried in an overlay node's payload.

    `op` semantics:
    - `prune`: drop `targets` from the effective view (payload unused).
    - `replace_text`: payload `{"text": ...}` replaces each target's text.
    - `boundary_shift`: payload `{"boundary_after": <left segment id>,
      "text": <moved text>, "direction": "push"|"pull"}` moves text across the
      boundary between two adjacent FIXED positions (push = from the end of the
      left unit to the start of the right; pull = the mirror). 1:1 alignment is
      maintained continuously — count and positions never change.
    """
    edit_id: str                                  # Carrying overlay node id (supersession anchor)
    op: str                                       # One of EDIT_OPS
    targets: List[str] = field(default_factory=list)   # Layer-0 segment node ids the edit applies to
    payload: Dict[str, Any] = field(default_factory=dict)  # Op-specific payload (see above)
    created_at: float = 0.0                       # Decision timestamp (application order + latest-wins tiebreak)

    def __post_init__(self):
        if self.op not in EDIT_OPS:
            raise SpineEditError(f"unknown spine-edit op: {self.op!r} (known: {EDIT_OPS})")

    def to_dict(self) -> Dict[str, Any]:  # Payload-ready dict
        """Serialize for carriage in an overlay node payload."""
        return {"edit_id": self.edit_id, "op": self.op, "targets": list(self.targets),
                "payload": dict(self.payload), "created_at": self.created_at}

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "SpineEdit":  # Reconstructed edit
        """Reconstruct from a payload dict."""
        return cls(edit_id=d["edit_id"], op=d["op"], targets=list(d.get("targets") or []),
                   payload=dict(d.get("payload") or {}), created_at=float(d.get("created_at") or 0.0))

In [ ]:
#| export
def resolve_active(
    edit_ids: Iterable[str],                      # Candidate overlay node ids
    supersedes_pairs: Iterable[Tuple[str, str]],  # (superseder_id, superseded_id) SUPERSEDES edges
) -> Set[str]:  # Active (non-superseded) ids
    """Resolve the active set under append-only supersession.

    An id is superseded iff it is the TARGET of any SUPERSEDES edge — chains
    resolve naturally (C supersedes B supersedes A leaves only C active), and
    nothing is ever mutated (the C16 semantics, now layer-owned).
    """
    superseded = {target for _, target in supersedes_pairs}
    return {eid for eid in edit_ids if eid not in superseded}

In [ ]:
#| export
def project_effective_spine(
    units: List[SpineUnit],   # Ordered layer-0 spine (immutable input)
    edits: List[SpineEdit],   # ACTIVE edits to apply (resolve supersession first)
) -> List[SpineUnit]:  # New effective spine (input never mutated)
    """Project the effective view: layer-0 + active edits, resolved at read time.

    Edits apply in (created_at, edit_id) order over the evolving text state, so
    later decisions see earlier ones' effects and replace_text latest-wins
    emerges from ordering. Prunes drop positions at the end (a boundary_shift
    or replace recorded before a later prune still applies cleanly).
    boundary_shift is STRICT: if the current text no longer carries the moved
    text verbatim at the boundary, the projection fails loudly rather than
    guessing (SpineEditError).
    """
    order = {u.id: i for i, u in enumerate(units)}
    texts = {u.id: u.text for u in units}
    pruned: Set[str] = set()

    for e in sorted(edits, key=lambda e: (e.created_at, e.edit_id)):
        if e.op == "prune":
            pruned.update(e.targets)
        elif e.op == "replace_text":
            new_text = e.payload.get("text", "")
            for t in e.targets:
                if t in texts:
                    texts[t] = new_text
        elif e.op == "boundary_shift":
            left = e.payload.get("boundary_after")
            moved = e.payload.get("text", "")
            direction = e.payload.get("direction", "push")
            if left not in order:
                raise SpineEditError(f"boundary_shift: unknown boundary_after {left!r}")
            idx = order[left]
            if idx + 1 >= len(units):
                raise SpineEditError("boundary_shift: no unit after the boundary")
            right = units[idx + 1].id
            if direction == "push":
                if not texts[left].endswith(moved):
                    raise SpineEditError(f"boundary_shift push: left text does not end with the moved text ({e.edit_id})")
                texts[left] = texts[left][: len(texts[left]) - len(moved)]
                texts[right] = moved + texts[right]
            elif direction == "pull":
                if not texts[right].startswith(moved):
                    raise SpineEditError(f"boundary_shift pull: right text does not start with the moved text ({e.edit_id})")
                texts[right] = texts[right][len(moved):]
                texts[left] = texts[left] + moved
            else:
                raise SpineEditError(f"boundary_shift: unknown direction {direction!r}")

    return [SpineUnit(u.id, texts[u.id]) for u in units if u.id not in pruned]

In [ ]:
# tests — vocabulary + supersession
try:
    SpineEdit("e0", "merge")
    raise AssertionError("expected SpineEditError")
except SpineEditError:
    pass

active = resolve_active(["a", "b", "c"], [("c", "a")])
assert active == {"b", "c"}, "superseded target excluded"
# chain: c supersedes b supersedes a -> only c active
active = resolve_active(["a", "b", "c"], [("b", "a"), ("c", "b")])
assert active == {"c"}

# round-trip
e = SpineEdit("e1", "replace_text", ["s1"], {"text": "fixed"}, created_at=2.0)
assert SpineEdit.from_dict(e.to_dict()) == e
print("edit vocab tests OK")

In [ ]:
# tests — projection: prune, replace latest-wins, ordering
spine = [SpineUnit("s1", "hello world"), SpineUnit("s2", ""), SpineUnit("s3", "the end")]
out = project_effective_spine(spine, [SpineEdit("p1", "prune", ["s2"], created_at=1.0)])
assert [u.id for u in out] == ["s1", "s3"], "pruned position dropped"
assert [u.text for u in spine] == ["hello world", "", "the end"], "input not mutated"

out = project_effective_spine(spine, [
    SpineEdit("r1", "replace_text", ["s1"], {"text": "first"}, created_at=1.0),
    SpineEdit("r2", "replace_text", ["s1"], {"text": "second"}, created_at=2.0),
])
assert out[0].text == "second", "latest replace wins via ordering"
print("projection prune/replace tests OK")

In [ ]:
# tests — boundary_shift push/pull + strict mismatch + composition
spine = [SpineUnit("s1", "the art of war "), SpineUnit("s2", "is of vital importance")]
out = project_effective_spine(spine, [SpineEdit("b1", "boundary_shift", [],
    {"boundary_after": "s1", "text": "war ", "direction": "push"}, created_at=1.0)])
assert out[0].text == "the art of " and out[1].text == "war is of vital importance"

out = project_effective_spine(spine, [SpineEdit("b2", "boundary_shift", [],
    {"boundary_after": "s1", "text": "is ", "direction": "pull"}, created_at=1.0)])
assert out[0].text == "the art of war is " and out[1].text == "of vital importance"

# strict: moved text absent at the boundary -> loud
try:
    project_effective_spine(spine, [SpineEdit("b3", "boundary_shift", [],
        {"boundary_after": "s1", "text": "XYZ", "direction": "push"}, created_at=1.0)])
    raise AssertionError("expected SpineEditError")
except SpineEditError:
    pass

# composition: replace then shift sees the replaced text
out = project_effective_spine(spine, [
    SpineEdit("r1", "replace_text", ["s1"], {"text": "AB CD "}, created_at=1.0),
    SpineEdit("b4", "boundary_shift", [], {"boundary_after": "s1", "text": "CD ", "direction": "push"}, created_at=2.0),
])
assert out[0].text == "AB " and out[1].text == "CD is of vital importance"
print("boundary_shift tests OK")